## 데이터 수집 자동화

### 안정적 수집과 저장

#### 수집 오류 유형

- 요소 없음 : CSS선택자가 틀렸거나 화면에 없음 > 예외처리
- 로딩 지연 : 페이지가 늦게 뜸 > 명시적 대기
- 클릭 실패 : 버튼이 아직 클릭불가 > 명시적 대기
- 데이터 누락 : 일부 항목이 페이지에 없음 > 기본값 처리
- 중복 수집 : 같은 데이터가 포함 > 판다스 중복처리
- 저장 실패 : 파일 존재여부, 권한여부 > 파일명 변경, 예외처리 

In [ ]:
try:                                            # 실행 중 오류가 날 것 같은 소스코드 (숫자입력시 정상, 문자입력시 오류)
    price = input('가격입력 > ')
    print(int(price))
except Exception as e:                          # 실행 중 오류가 발생하면 처리할 부분
    print(f'가격은 숫자만 입력할 수 있습니다')  
finally:                                        # 실행 중 오류가 나든 안나든 무조건 실행하는 부분
    print('프로그램종료') # 예외

가격은 숫자만 입력할 수 있습니다
프로그램종료


In [10]:
from selenium import webdriver

In [ ]:
driver = webdriver.Chrome()

try : 
    driver.get('https://www.naver.com')    # 웹사이트 오픈
    print(driver.title )                   # 웹사이트의 head 영역에있는 텝이름(title) 이름 가져오기

except Exception as e:
    print(f'실행 중 오류발생! : {e.args}')  # 예외 발생시 표시

finally:                                  
    driver.quit()                          # 실행 후 종료

NAVER


#### find_element의 문제점
- find_elements() 실행 중 오류가 발생하지 않음
- find_element() 실행 중 오류가 발생할 가능성이 높음
    - 함수를 만드는게 유리

In [14]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
def get_text_safe(parent, selector, default=''):
    try : 
        return parent.find_element(By.CSS_SELECTOR. selector).text
    except:
        return default              # 내가 지정한 기본값이 그대로 리턴

In [16]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com/')

wait.until(
    EC.visibility_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="56faf75d3cf6df8d8eb2c8eb9ff8ab0e", element="f.C048FEA18F67613DBBA48508B028B020.d.2A98DF11D457C19B88F50205CEBBFD9A.e.75")>

In [18]:
get_text_safe(driver, 'div.item > a.txt', 'None')

'None'

#### 명시적 대기와 타임아웃
- 명시적 대기를 하더라도 시간을 초과하면 실행 중 오류가 발생
- 실행 중 오류를 잡으려면 try, except 사용 필요

In [22]:
from selenium.common.exceptions import TimeoutException

try: 
    driver.get('https://www.naver.com/')
    wait.until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, '#footer'))
    )
    print('로딩완료')
except TimeoutException as e:
    print('로딩 실패! 타임오버')

로딩완료


#### 버튼 클릭이 가능상태까지 대기
- 버튼, 링크가 화면에 나타나도 js가 로딩완료 될때까지 클릭 못하는 경우가 발생할 수 있음
- 클릭상태까지 대기

In [25]:
enable_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, '#account a'))
)

enable_button.click()

#### 로그파일 저장
- 화면에 출력보다 로그파일 저장이 효율적

In [31]:
import logging

logging.basicConfig(
    filename='scraping.log',
    level=logging.INFO,              # DEBUG,INFO,WARN,ERROR,WARNING
    format="%(asctime)s %(levelname)s %(message)",
    encoding='utf-8'
)

In [32]:
logging.info('수집시작')
logging.info('상품데이터수집완료')
logging.error('수집 중 오류발생!!')

--- Logging error ---
Traceback (most recent call last):
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 1151, in emit
    msg = self.format(record)
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 999, in format
    return fmt.format(record)
           ~~~~~~~~~~^^^^^^^^
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 715, in format
    s = self.formatMessage(record)
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 684, in formatMessage
    return self._style.format(record)
           ~~~~~~~~~~~~~~~~~~^^^^^^^^
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 472, in format
    return self._format(record)
           ~~~~~~~~~~~~^^^^^^^^
  File "c:\Program Files\Python314\Lib\logging\__init__.py", line 468, in _format
    return self._fmt % values
           ~~~~~~~~~~^~~~~~~~
ValueError: incomplete format
Call stack:
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", 

#### DB에 저장
- 이전 저장한 다나와 데이터 사용

In [33]:
df = pd.read_csv('../data/danawa_게이밍PC.csv', encoding='utf-8')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     900 non-null    int64  
 1   index          900 non-null    int64  
 2   computer_name  900 non-null    str    
 3   computer_spec  900 non-null    str    
 4   price_spec     899 non-null    str    
 5   price          899 non-null    float64
dtypes: float64(1), int64(2), str(3)
memory usage: 42.3 KB


#### 스케쥴링
> 정해진 시간마다 반복수행하도록 하는 것
- OS 마다 방식이 상이
    - Windows : 작업 스케줄러
    - Linux :  cron
    - Python : schedule 라이브러리로 가능

#### 윈도우 작업 스케쥴러

- 윈도우키 > 작업 스케줄러 검색 후 실행
- `기본작업만들기...` 클릭
- 기본 작업만들기 탭 > 이름과 설명(옵션) 추가
- 트리거 > 작업을 매일로 선택
    - 작업 내용은 시작시간과 반복 횟수 등 상황에 맞게 변경
- 작업 > 프로그램시작
    - 프로그램스크립트 : .bat 파일 위치
    - 시작 위치 : D:\sourcebank\ai-datapipeline\chapt04 와 같이 경로 입력
- 속성 창 오픈
    - 보안 옵션 : Administrators(관리자 그룹)을 선택하고 사용자 변경
    - 가장 높은 수준의 권한으로 실행